In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

START_DATE = "2006-01-01"
END_DATE = "2026-10-23"

BASE_DIR = Path(r"D:\Project_Github\gold assisstant\dataset\news")
RAW_DIR = BASE_DIR / "raw"
PROCESSED_DIR = BASE_DIR / "processed"
AUDIT_DIR = BASE_DIR / "audit"

for path in [RAW_DIR, PROCESSED_DIR, AUDIT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Date range:", START_DATE, "to", END_DATE)
print("Base directory:", BASE_DIR.resolve())

Date range: 2006-01-01 to 2026-10-23
Base directory: D:\Project_Github\gold assisstant\dataset\news


In [2]:
sources = [
    {
        "source_id": "gdelt_events",
        "source_name": "GDELT Event Database",
        "source_type": "structured_events",
        "languages": "multilingual",
        "historical_start": "1979-01-01",
        "historical_end": None,
        "access_type": "open_data",
        "status": "candidate",
        "coverage_verified": False,
        "license_reviewed": False,
        "url": "https://www.gdeltproject.org/",
    },
    {
        "source_id": "gdelt_doc",
        "source_name": "GDELT DOC API",
        "source_type": "article_search",
        "languages": "multilingual",
        "historical_start": None,
        "historical_end": None,
        "access_type": "public_api",
        "status": "recent_search_only",
        "coverage_verified": False,
        "license_reviewed": False,
        "url": "https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/",
    },
    {
        "source_id": "newsapi",
        "source_name": "NewsAPI",
        "source_type": "article_metadata",
        "languages": "multiple",
        "historical_start": None,
        "historical_end": None,
        "access_type": "api_key_and_plan",
        "status": "plan_review_required",
        "coverage_verified": False,
        "license_reviewed": False,
        "url": "https://newsapi.org/pricing",
    },
    {
        "source_id": "newsdata_archive",
        "source_name": "NewsData.io Archive",
        "source_type": "article_metadata",
        "languages": "multiple",
        "historical_start": None,
        "historical_end": None,
        "access_type": "paid_api",
        "status": "plan_review_required",
        "coverage_verified": False,
        "license_reviewed": False,
        "url": "https://newsdata.io/documentation/news-archive-api",
    },
    {
        "source_id": "nbt_open_api",
        "source_name": "NBT Open API",
        "source_type": "thai_news",
        "languages": "th",
        "historical_start": None,
        "historical_end": None,
        "access_type": "registration_or_api_access",
        "status": "historical_coverage_unknown",
        "coverage_verified": False,
        "license_reviewed": False,
        "url": "https://openapi-thainews.prd.go.th/",
    },
]

df_sources = pd.DataFrame(sources)

df_sources.to_csv(
    AUDIT_DIR / "source_registry.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_sources)

,source_id,source_name,source_type,languages,historical_start,historical_end,access_type,status,coverage_verified,license_reviewed,url
0,gdelt_events,GDELT Event Database,structured_events,multilingual,1979-01-01,None,open_data,candidate,False,False,https://www.gdeltproject.org/
1,gdelt_doc,GDELT DOC API,article_search,multilingual,NaN,None,public_api,recent_search_only,False,False,https://blog.gdeltproject.org/gdelt-doc-2-0-ap...
2,newsapi,NewsAPI,article_metadata,multiple,NaN,None,api_key_and_plan,plan_review_required,False,False,https://newsapi.org/pricing
3,newsdata_archive,NewsData.io Archive,article_metadata,multiple,NaN,None,paid_api,plan_review_required,False,False,https://newsdata.io/documentation/news-archive...
4,nbt_open_api,NBT Open API,thai_news,th,NaN,None,registration_or_api_access,historical_coverage_unknown,False,False,https://openapi-thainews.prd.go.th/


In [3]:
NEWS_COLUMNS = [
    "article_id",
    "source_id",
    "source_article_id",
    "source_url",
    "title",
    "language",
    "published_at_raw",
    "published_at_utc",
    "published_timezone",
    "event_time_raw",
    "event_time_utc",
    "first_seen_at_utc",
    "retrieved_at_utc",
    "raw_payload_hash",
    "content_hash",
    "timestamp_quality",
    "content_license_status",
]

df_news_empty = pd.DataFrame(columns=NEWS_COLUMNS)

print("Number of schema fields:", len(NEWS_COLUMNS))
display(df_news_empty.dtypes.to_frame("dtype"))

Number of schema fields: 17


,dtype
article_id,object
source_id,object
source_article_id,object
source_url,object
title,object
language,object
published_at_raw,object
published_at_utc,object
published_timezone,object
event_time_raw,object


In [4]:
def parse_timestamp_utc(series):
    """
    Convert timestamps to UTC.
    Naive timestamps are treated as unknown timezone and flagged
    separately; they are not automatically assumed to be UTC.
    """
    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed


def audit_timestamps(df, column="published_at_raw"):
    out = df.copy()

    out["published_at_utc"] = parse_timestamp_utc(out[column])

    out["timestamp_missing"] = out[column].isna()
    out["timestamp_unparseable"] = (
        out[column].notna()
        & out["published_at_utc"].isna()
    )

    out["timestamp_out_of_range"] = (
        out["published_at_utc"].notna()
        & (
            (out["published_at_utc"] < pd.Timestamp("2006-01-01", tz="UTC"))
            | (out["published_at_utc"] >= pd.Timestamp("2026-10-24", tz="UTC"))
        )
    )

    return out

In [5]:
def audit_duplicates(df):
    out = df.copy()

    # Normalize URL conservatively; do not remove query parameters
    # because they may occasionally be meaningful to the source.
    out["url_key"] = (
        out["source_url"]
        .astype("string")
        .str.strip()
        .str.rstrip("/")
    )

    out["title_key"] = (
        out["title"]
        .astype("string")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    out["duplicate_url"] = (
        out["url_key"].notna()
        & out.duplicated("url_key", keep=False)
    )

    out["duplicate_source_id"] = (
        out["source_article_id"].notna()
        & out.duplicated(
            ["source_id", "source_article_id"],
            keep=False
        )
    )

    out["duplicate_title_source_day"] = (
        out["published_at_utc"].notna()
        & out["title_key"].notna()
        & out.duplicated(
            [
                "source_id",
                "title_key",
                out["published_at_utc"].dt.strftime("%Y-%m-%d"),
            ],
            keep=False
        )
    )

    return out

In [6]:
def build_coverage_report(df):
    required = ["source_id", "language", "published_at_utc"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    x = df.copy()
    x["published_at_utc"] = pd.to_datetime(
        x["published_at_utc"], errors="coerce", utc=True
    )

    x["year"] = x["published_at_utc"].dt.year

    x = x[
        x["year"].between(2006, 2026)
    ]

    coverage = (
        x.groupby(["source_id", "language", "year"])
         .agg(
             article_count=("article_id", "nunique")
                 if "article_id" in x.columns
                 else ("published_at_utc", "size"),
             first_publication=("published_at_utc", "min"),
             last_publication=("published_at_utc", "max"),
             days_with_news=("published_at_utc", lambda s: s.dt.date.nunique()),
         )
         .reset_index()
         .sort_values(["source_id", "language", "year"])
    )

    return coverage

In [7]:
from urllib.parse import urlparse
import re

def normalize_source_url(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # Convert Markdown link: [label](https://example.com)
    match = re.fullmatch(r"\[.*?\]\((https?://.*?)\)", value)
    if match:
        value = match.group(1)

    # Validate URL structure
    parsed = urlparse(value)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return None

    return value

df_sources["url_original"] = df_sources["url"]
df_sources["url"] = df_sources["url"].apply(normalize_source_url)

display(df_sources[["source_id", "source_name", "url"]])

assert df_sources["url"].notna().all(), "พบ URL ที่ไม่ถูกต้อง"
assert df_sources["url"].str.startswith(("http://", "https://")).all()

,source_id,source_name,url
0,gdelt_events,GDELT Event Database,https://www.gdeltproject.org/
1,gdelt_doc,GDELT DOC API,https://blog.gdeltproject.org/gdelt-doc-2-0-ap...
2,newsapi,NewsAPI,https://newsapi.org/pricing
3,newsdata_archive,NewsData.io Archive,https://newsdata.io/documentation/news-archive...
4,nbt_open_api,NBT Open API,https://openapi-thainews.prd.go.th/


In [8]:
display(
    df_sources[
        ["source_id", "source_name", "url"]
    ].to_dict(orient="records")
)

[{'source_id': 'gdelt_events',
  'source_name': 'GDELT Event Database',
  'url': 'https://www.gdeltproject.org/'},
 {'source_id': 'gdelt_doc',
  'source_name': 'GDELT DOC API',
  'url': 'https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/'},
 {'source_id': 'newsapi',
  'source_name': 'NewsAPI',
  'url': 'https://newsapi.org/pricing'},
 {'source_id': 'newsdata_archive',
  'source_name': 'NewsData.io Archive',
  'url': 'https://newsdata.io/documentation/news-archive-api'},
 {'source_id': 'nbt_open_api',
  'source_name': 'NBT Open API',
  'url': 'https://openapi-thainews.prd.go.th/'}]

In [9]:
registry_path = AUDIT_DIR / "source_registry.csv"

df_sources.to_csv(
    registry_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", registry_path.resolve())
print("Rows:", len(df_sources))

Saved: D:\Project_Github\gold assisstant\dataset\news\audit\source_registry.csv
Rows: 5


# GDELT

In [14]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

GDELT_INDEX = "https://data.gdeltproject.org/events/"

response = requests.get(
    GDELT_INDEX,
    timeout=30,
    headers={"User-Agent": "QuantResearchDataset/1.0"}
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.endswith(".zip"):
        links.append(href)

df_gdelt_files = pd.DataFrame({"filename": links})

patterns = [
    r"^2006\d{2}\.export\.CSV\.zip$",
    r"^2013\d{2}\d{2}\.export\.CSV\.zip$",
    r"^2026\d{4}\.export\.CSV\.zip$",
    r"^GDELT\.MASTERREDUCEDV2\.1979-2013\.zip$",
]


def classify_file(filename):
    if filename == "GDELT.MASTERREDUCEDV2.1979-2013.zip":
        return "historical_master"

    if re.fullmatch(r"\d{8}\.export\.CSV\.zip", filename):
        return "daily"

    if re.fullmatch(r"\d{6}\.zip", filename):
        return "monthly"

    if re.fullmatch(r"\d{4}\.zip", filename):
        return "yearly"

    return "other"


files["file_type_corrected"] = files["filename"].apply(classify_file)

# แยกรายงานโดยไม่นับชื่อไฟล์ซ้ำ
files_unique = files.drop_duplicates(subset="filename").copy()

print("Unique ZIP filenames:", len(files_unique))

display(
    files_unique["file_type_corrected"]
    .value_counts()
    .rename_axis("file_type")
    .reset_index(name="count")
)

print("\nFiles not classified:")
display(
    files_unique.loc[
        files_unique["file_type_corrected"] == "other",
        ["filename"]
    ].sort_values("filename")
)


HTTP status: 200
Content type: text/html
Unique ZIP filenames: 5046


,file_type,count
0,daily,4931
1,monthly,87
2,yearly,27
3,historical_master,1



Files not classified:


,filename


In [15]:
daily_names = files_unique.loc[
    files_unique["file_type_corrected"] == "daily",
    "filename"
]

daily_dates = pd.to_datetime(
    daily_names.str.extract(r"^(\d{8})")[0],
    format="%Y%m%d",
    errors="coerce"
).dropna().dt.normalize()

start = pd.Timestamp("2013-04-01")
end = pd.Timestamp("2026-10-22")

expected = pd.date_range(start, end, freq="D")
observed = pd.DatetimeIndex(daily_dates.unique())

missing = expected.difference(observed)

print("Expected daily files:", len(expected))
print("Observed daily files:", len(
    observed[(observed >= start) & (observed <= end)]
))
print("Missing dates:", len(missing))
print(missing.strftime("%Y-%m-%d").tolist())

print("\nDaily coverage by year:")
coverage = pd.DataFrame({"date": expected})
coverage["year"] = coverage["date"].dt.year
coverage["has_file"] = coverage["date"].isin(observed)

display(
    coverage.groupby("year")["has_file"]
    .agg(expected_days="size", available_days="sum")
    .assign(
        missing_days=lambda x: x["expected_days"] - x["available_days"]
    )
)

Expected daily files: 4953
Observed daily files: 4931
Missing dates: 22
['2014-01-23', '2014-01-24', '2014-01-25', '2014-03-19', '2025-06-14', '2025-06-15', '2025-06-16', '2025-06-17', '2025-06-18', '2025-06-19', '2025-06-20', '2025-06-21', '2025-06-22', '2025-06-23', '2025-06-24', '2025-06-25', '2025-06-26', '2025-06-27', '2025-06-28', '2025-06-29', '2025-06-30', '2025-07-01']

Daily coverage by year:


,expected_days,available_days,missing_days
year,,,
2013,275,275,0
2014,365,361,4
2015,365,365,0
2016,366,366,0
2017,365,365,0
2018,365,365,0
2019,365,365,0
2020,366,366,0
2021,365,365,0


In [16]:
# Cell 17 — Verify missing GDELT files directly

import requests
import pandas as pd

GDELT_INDEX = "https://data.gdeltproject.org/events/"

headers = {
    "User-Agent": "QuantResearchDataset/1.0"
}

check_results = []

for date in missing:
    date_str = pd.Timestamp(date).strftime("%Y%m%d")
    filename = f"{date_str}.export.CSV.zip"
    url = GDELT_INDEX + filename

    try:
        # ตรวจสอบผ่าน HEAD ก่อน
        response = requests.head(
            url,
            headers=headers,
            timeout=20,
            allow_redirects=True
        )

        # หาก HEAD ไม่รองรับ ให้ลอง GET แบบ stream
        if response.status_code in (403, 405, 501):
            response.close()
            response = requests.get(
                url,
                headers={**headers, "Range": "bytes=0-0"},
                timeout=20,
                allow_redirects=True,
                stream=True
            )

        status = response.status_code
        content_length = response.headers.get("Content-Length")

        check_results.append({
            "date": date.strftime("%Y-%m-%d"),
            "filename": filename,
            "http_status": status,
            "content_length": content_length,
            "direct_url_exists": status in (200, 206),
            "url": url
        })

        response.close()

    except requests.RequestException as exc:
        check_results.append({
            "date": date.strftime("%Y-%m-%d"),
            "filename": filename,
            "http_status": None,
            "content_length": None,
            "direct_url_exists": None,
            "url": url,
            "error": str(exc)
        })

df_missing_checks = pd.DataFrame(check_results)

display(df_missing_checks)

print("\nHTTP status summary:")
display(
    df_missing_checks["http_status"]
    .value_counts(dropna=False)
    .rename_axis("http_status")
    .reset_index(name="count")
)

,date,filename,http_status,content_length,direct_url_exists,url
0,2014-01-23,20140123.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20140123....
1,2014-01-24,20140124.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20140124....
2,2014-01-25,20140125.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20140125....
3,2014-03-19,20140319.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20140319....
4,2025-06-14,20250614.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250614....
5,2025-06-15,20250615.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250615....
6,2025-06-16,20250616.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250616....
7,2025-06-17,20250617.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250617....
8,2025-06-18,20250618.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250618....
9,2025-06-19,20250619.export.CSV.zip,404,0,False,https://data.gdeltproject.org/events/20250619....



HTTP status summary:


,http_status,count
0,404,22


In [17]:
# Cell 18 — Save missing-file audit

audit_cols = [
    "date",
    "filename",
    "http_status",
    "direct_url_exists",
    "url",
]

df_missing_audit = df_missing_checks[audit_cols].copy()

df_missing_audit["source_id"] = "gdelt_events"
df_missing_audit["status"] = "missing_from_expected_daily_url"
df_missing_audit["checked_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
df_missing_audit["note"] = (
    "Direct URL returned 404; availability in alternative archives "
    "has not been verified."
)

audit_path = AUDIT_DIR / "gdelt_missing_daily_files.csv"
df_missing_audit.to_csv(audit_path, index=False)

print("Saved:", audit_path)
print("Rows:", len(df_missing_audit))

Saved: D:\Project_Github\gold assisstant\dataset\news\audit\gdelt_missing_daily_files.csv
Rows: 22


In [18]:
from pathlib import Path
import requests
import pandas as pd
import time

GDELT_INDEX = "https://data.gdeltproject.org/events/"
ZIP_DIR = RAW_DIR / "gdelt_events"
ZIP_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = AUDIT_DIR / "gdelt_download_manifest.csv"

# เลือกเฉพาะไฟล์รายวันในช่วง 2013-04-01 ถึง 2026-10-22
daily_files = (
    files.loc[
        files["file_type_corrected"] == "daily",
        "filename"
    ]
    .drop_duplicates()
    .sort_values()
)

daily_files = daily_files[
    daily_files.str.extract(r"^(\d{8})")[0].between(
        "20130401", "20261022"
    )
].tolist()

# อ่าน manifest เดิม เพื่อไม่ดาวน์โหลดไฟล์ที่สำเร็จแล้วซ้ำ
if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
else:
    manifest = pd.DataFrame(columns=[
        "filename", "url", "status",
        "http_status", "size_bytes", "sha256"
    ])

completed = set(
    manifest.loc[manifest["status"] == "downloaded", "filename"]
)

session = requests.Session()
session.headers.update({
    "User-Agent": "QuantResearchDataset/1.0"
})

results = []

for i, filename in enumerate(daily_files, start=1):
    if filename in completed:
        continue

    url = GDELT_INDEX + filename
    destination = ZIP_DIR / filename
    temp_path = destination.with_suffix(".zip.part")

    try:
        with session.get(url, stream=True, timeout=(15, 120)) as response:
            if response.status_code == 404:
                results.append({
                    "filename": filename,
                    "url": url,
                    "status": "missing_404",
                    "http_status": 404,
                    "size_bytes": None,
                    "sha256": None,
                })
                continue

            response.raise_for_status()

            import hashlib
            digest = hashlib.sha256()
            size = 0

            with open(temp_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    f.write(chunk)
                    digest.update(chunk)
                    size += len(chunk)

        # ตรวจว่าเป็น ZIP ที่อ่านได้ก่อนเปลี่ยนชื่อเป็นไฟล์จริง
        import zipfile
        if not zipfile.is_zipfile(temp_path):
            temp_path.unlink(missing_ok=True)
            results.append({
                "filename": filename,
                "url": url,
                "status": "invalid_zip",
                "http_status": response.status_code,
                "size_bytes": size,
                "sha256": digest.hexdigest(),
            })
            continue

        temp_path.replace(destination)

        results.append({
            "filename": filename,
            "url": url,
            "status": "downloaded",
            "http_status": response.status_code,
            "size_bytes": size,
            "sha256": digest.hexdigest(),
        })

    except Exception as exc:
        temp_path.unlink(missing_ok=True)
        results.append({
            "filename": filename,
            "url": url,
            "status": f"error: {type(exc).__name__}: {exc}",
            "http_status": None,
            "size_bytes": None,
            "sha256": None,
        })

    # บันทึกความคืบหน้าเป็นระยะ
    if len(results) >= 10:
        manifest = pd.concat(
            [manifest, pd.DataFrame(results)],
            ignore_index=True
        ).drop_duplicates("filename", keep="last")

        manifest.to_csv(MANIFEST_PATH, index=False)
        results = []

    if i % 100 == 0:
        print(f"Checked {i}/{len(daily_files)} files")

    time.sleep(0.1)

if results:
    manifest = pd.concat(
        [manifest, pd.DataFrame(results)],
        ignore_index=True
    ).drop_duplicates("filename", keep="last")

manifest.to_csv(MANIFEST_PATH, index=False)

print("ZIP directory:", ZIP_DIR.resolve())
print("Manifest:", MANIFEST_PATH.resolve())
display(manifest["status"].value_counts())

KeyboardInterrupt: 

ประมาณขนาดข้อมูล GDELT ปี 2006–2026

ช่วงที่เราต้องการคือ 1 มกราคม 2006 – 22 ตุลาคม 2026 รวม 7,599 วัน (ประมาณ 20.8 ปี)

หาก 87.3 MB เป็นข้อมูล 41 วัน:


*   **ค่าเฉลี่ยต่อวัน:** 
    $$\frac{87.3}{41} \approx 2.13 \text{ MB}$$

*   **ขนาดประมาณทั้งช่วง:** 
    $$2.13 \times 7,599 \approx 16,183 \text{ MB} \approx 15.8 \text{ GiB}$$


ขนาดประมาณสำหรับช่วงปี 2006–2026



# ให้ llm ดึงข้อมูลข่าวเอง